# NEU-ESC — chạy run thật trên Kaggle T4 x2

Chạy các setup cho danh sách `SEEDS`, mỗi GPU một run. Nên bấm **Save Version → Save & Run All (Commit)** để notebook chạy nền (tối đa 12 giờ/session).
Run nào đã có `metrics.json` trạng thái `COMPLETED` trong `OUT_DIR` sẽ được bỏ qua, nên có thể chạy lại queue sau khi session bị ngắt.

Kết quả mỗi run: `config.json` (gồm hash của code), `epochs.jsonl`, `metrics.json`, `logits.npz`, `predictions.csv` (dự đoán theo ID mẫu). Không lưu checkpoint.
Sau khi xong, tải thư mục `runs/` từ tab Output (hoặc tạo Kaggle Dataset từ output) để gộp với các session khác.

**Điều kiện đi tiếp sang seed khác** là cell kiểm tra kỹ thuật bên dưới báo PASS cho mọi run. Bảng so với Table 6 chỉ để báo cáo, không dùng để duyệt.

**Cài đặt notebook:** Accelerator = **GPU T4 x2**, Internet = **On**.

In [ ]:
import glob, os, shutil, subprocess

WORK = "/kaggle/working"
CODE_DIR = f"{WORK}/neu-esc"
DATA_DIR = f"{WORK}/data"
# Cách 1: upload thư mục src_baseline/ của repo thành một Kaggle Dataset rồi "Add Input" vào notebook.
# Cách 2: clone từ GitHub (repo phải public, hoặc dùng token qua Kaggle Secrets).
REPO_URL = "https://github.com/AI-Peak/neu-esc-reproduction.git"
BRANCH = "main"

uploaded = glob.glob("/kaggle/input/**/neu_esc_kaggle.py", recursive=True)
os.makedirs(f"{CODE_DIR}/src_baseline", exist_ok=True)
if uploaded:
    for path in glob.glob(os.path.join(os.path.dirname(uploaded[0]), "*.py")):
        shutil.copy(path, f"{CODE_DIR}/src_baseline/")
    print("code from dataset:", os.path.dirname(uploaded[0]))
else:
    shutil.rmtree(CODE_DIR)
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, CODE_DIR], check=True)
    print("code from git:", subprocess.check_output(["git", "-C", CODE_DIR, "rev-parse", "HEAD"], text=True).strip())
RUNNER = f"{CODE_DIR}/src_baseline/neu_esc_kaggle.py"
QUEUE = f"{CODE_DIR}/src_baseline/run_queue.py"
AGGREGATE = f"{CODE_DIR}/src_baseline/aggregate.py"
CHECK = f"{CODE_DIR}/src_baseline/check_runs.py"
print(sorted(os.listdir(f"{CODE_DIR}/src_baseline")))

In [ ]:
# Giữ transformers 4.x (pipeline đã kiểm với 4.x); sentencepiece cần cho tokenizer ViSoBERT.
!pip install -q "transformers>=4.44,<5" sentencepiece
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!python -c "import torch, transformers, sklearn; print('torch', torch.__version__, '| transformers', transformers.__version__, '| sklearn', sklearn.__version__, '| GPUs', torch.cuda.device_count())"

In [ ]:
# Tải 6 CSV của tác giả (commit b5637cc) và kiểm SHA-256.
!python {RUNNER} fetch --data-dir {DATA_DIR}

In [ ]:
SEEDS = [42]                      # mỗi session nên chạy 1 seed x 8 setup
BACKBONES = ["phobert", "visobert"]
SETUPS = ["st_sentiment", "st_topic", "mtl", "mtl_smart"]
OUT_DIR = f"{WORK}/runs"
seeds, backbones, setups = " ".join(map(str, SEEDS)), " ".join(BACKBONES), " ".join(SETUPS)
!python {QUEUE} --data-dir {DATA_DIR} --out-dir {OUT_DIR} --seeds {seeds} --backbones {backbones} --setups {setups}

In [ ]:
# Kiểm tra kỹ thuật (không so với paper): run xong, loss giảm, hơn mức đoán lớp đa số, đủ dự đoán, cùng phiên bản code.
!python {CHECK} {OUT_DIR} --seeds {seeds} --backbones {backbones} --setups {setups}

In [ ]:
# Bảng tạm cho các run trong session này, chỉ để báo cáo (gộp nhiều session: chạy aggregate.py trên tất cả thư mục runs/).
!python {AGGREGATE} {OUT_DIR} --out {WORK}/summary